# CECS 456 — minimal CNN

Data → model → training → evaluation, without the walkthrough.

In [ ]:
import io
import random
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets
from torchvision.transforms import v2

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# Dataset: unpacked locally, downloaded once if missing.
data_root = Path("archive")
if not data_root.exists():
    url = ("https://drive.google.com/uc?export=download&id="
           "1hsJTZwrmPzQW0KoiyLmf_M-TjS71HB7_&confirm=t")
    urllib.request.urlretrieve(url, "archive.zip")
    zipfile.ZipFile("archive.zip").extractall(".")
    print("downloaded")

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

full_train = datasets.ImageFolder(data_root / "train", transform=transform)
test_set = datasets.ImageFolder(data_root / "test", transform=transform)

# 90k / 10k split, then a 5k training slice for a fast run.
train_set, val_set = random_split(
    full_train, [90000, 10000], generator=torch.Generator().manual_seed(SEED))
SUBSET = 5000
idx = torch.randperm(len(train_set), generator=torch.Generator().manual_seed(SEED))[:SUBSET]

train_loader = DataLoader(Subset(train_set, idx), batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=512)
test_loader = DataLoader(test_set, batch_size=512)
print(len(idx), len(val_set), len(test_set), full_train.class_to_idx)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.head(self.features(x))


model = CNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(sum(p.numel() for p in model.parameters()), "parameters")

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = total_correct = n = 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels.float())
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            n += labels.size(0)
    return total_loss / n, total_correct / n


for epoch in range(1, 6):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader)
    print(f"epoch {epoch}  train {train_loss:.3f}/{train_acc:.3f}  "
          f"val {val_loss:.3f}/{val_acc:.3f}")

In [ ]:
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, target in loader:
            probs += torch.sigmoid(model(images.to(device))).squeeze(1).cpu().tolist()
            labels += target.tolist()
    return np.array(labels), np.array(probs)


y_true, y_prob = predict(model, test_loader)
y_pred = (y_prob >= 0.5).astype(int)
print(f"accuracy {np.mean(y_pred == y_true):.4f}  "
      f"F1 {f1_score(y_true, y_pred):.4f}  "
      f"ROC-AUC {roc_auc_score(y_true, y_prob):.4f}")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# Same weights, degraded test images.
def jpeg(pil_img, quality=20):
    buf = io.BytesIO()
    pil_img.convert("RGB").save(buf, "JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def degraded_loader(pil_fn=None, tensor_fn=None):
    steps = [pil_fn] if pil_fn else []
    steps += [v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]
    steps += [v2.Lambda(tensor_fn)] if tensor_fn else []
    steps += [v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    ds = datasets.ImageFolder(data_root / "test", transform=v2.Compose(steps))
    return DataLoader(ds, batch_size=512)


noise = lambda x: torch.clamp(x + torch.randn_like(x) * 0.1, 0, 1)
blur = lambda x: v2.functional.gaussian_blur(x, kernel_size=3, sigma=1.0)

conditions = {
    "clean": test_loader,
    "jpeg q=20": degraded_loader(pil_fn=jpeg),
    "blur 3x3": degraded_loader(tensor_fn=blur),
    "noise 0.1": degraded_loader(tensor_fn=noise),
}
for name, loader in conditions.items():
    yt, yp = predict(model, loader)
    print(f"{name:<10} {np.mean((yp >= 0.5) == yt):.4f}")